In [1]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
from src.features import extract_features, get_host

df = pd.read_csv("../data/PhiUSIIL_Phishing_URL_Dataset.csv", usecols=["URL", "label"])
df = df.drop_duplicates(subset="URL").copy()
df["phish"] = (df["label"] == 0).astype(int)   # 1 = phishing (positive class)
df["host"] = df["URL"].map(get_host)

feats = pd.DataFrame([extract_features(u) for u in df["URL"]], index=df.index)
data = pd.concat([df[["host", "phish"]], feats], axis=1)
print(data.shape)
print("Unique hosts:", data["host"].nunique())

(235370, 17)
Unique hosts: 219094


In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Hosts that appear with both labels would confuse the model
conflict = data.groupby("host")["phish"].nunique()
conflict = conflict[conflict > 1].index
print("Hosts with both labels:", len(conflict))
data = data[~data["host"].isin(conflict)]

X = data.drop(columns=["host", "phish"])
y = data["phish"]
groups = data["host"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Phishing share train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))
print("Hosts shared between train and test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Hosts with both labels: 58
Train: (188682, 15) Test: (46529, 15)
Phishing share train/test: 0.429 0.42
Hosts shared between train and test: 0


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

for name, m in {"Logistic Regression": LogisticRegression(max_iter=2000),
                "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=1, random_state=42)}.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    print(name)
    print(classification_report(y_test, pred, target_names=["legitimate", "phishing"], digits=3))
    print(confusion_matrix(y_test, pred, labels=[1, 0]), "\n")

Logistic Regression
              precision    recall  f1-score   support

  legitimate      0.783     0.965     0.864     27010
    phishing      0.928     0.630     0.751     19519

    accuracy                          0.824     46529
   macro avg      0.856     0.798     0.808     46529
weighted avg      0.844     0.824     0.817     46529

[[12306  7213]
 [  956 26054]] 

Random Forest
              precision    recall  f1-score   support

  legitimate      0.792     0.956     0.866     27010
    phishing      0.915     0.652     0.762     19519

    accuracy                          0.829     46529
   macro avg      0.854     0.804     0.814     46529
weighted avg      0.844     0.829     0.823     46529

[[12736  6783]
 [ 1179 25831]] 

